# AI Documentation Assistant — Test Notebook
Tests the full pipeline end to end: load → chunk → embed → store → retrieve → answer.

## Step 1 — Install Packages

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic langchain-qdrant \
    langchain-community langchain-text-splitters qdrant-client \
    pypdf docx2txt beautifulsoup4 lxml requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


d:\AI Documentation Assistant\.venv\Scripts\python.exe: No module named pip


## Step 2 — Load API Keys

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY in .env"
assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY in .env"

print("API keys loaded successfully.")

API keys loaded successfully.


## Step 3 — Load Python Docs

In [3]:
from langchain_community.document_loaders import WebBaseLoader

SOURCES = [
    "https://docs.python.org/3/library/functions.html",
    "https://docs.python.org/3/library/stdtypes.html",
    "https://docs.python.org/3/library/exceptions.html",
]

docs = []
for url in SOURCES:
    loaded = WebBaseLoader(url).load()
    for doc in loaded:
        doc.metadata["source"] = url
    docs.extend(loaded)
    print(f"Loaded: {url} ({len(loaded)} page(s))")

print(f"\nTotal pages loaded: {len(docs)}")

C:\Users\ishas\AppData\Local\Temp\ipykernel_18856\2982618056.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded: https://docs.python.org/3/library/functions.html (1 page(s))
Loaded: https://docs.python.org/3/library/stdtypes.html (1 page(s))
Loaded: https://docs.python.org/3/library/exceptions.html (1 page(s))

Total pages loaded: 3


## Step 4 — Chunk Documents

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks   = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content[:300])

Total chunks: 519

Sample chunk:
Built-in Functions — Python 3.14.5 documentation




















































    Theme
    
Auto
Light
Dark



Previous topic
Introduction


Next topic
Built-in Constants



This page

Report a bug
Improve this page

Show source
        







Navigation


index

modules |




## Step 5 — Embed & Store in Qdrant

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME      = "docs_python"
EMBEDDING_DIMENSIONS = 1536

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Close any existing client to release the file lock before re-creating
try:
    client.close()
except Exception:
    pass

client = QdrantClient(path="../qdrant_storage")

# Create collection if it doesn't exist
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME not in existing:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIMENSIONS, distance=Distance.COSINE),
    )

vector_store = QdrantVectorStore(client=client, collection_name=COLLECTION_NAME, embedding=embeddings)
vector_store.add_documents(chunks)

total = client.get_collection(COLLECTION_NAME).points_count
print(f"Stored {total} vectors in '{COLLECTION_NAME}'")

## Step 6 — Test Retriever

In [6]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5})

query   = "What does the len() function do?"
results = retriever.invoke(query)

print(f"Query: {query}")
print(f"Retrieved {len(results)} chunks\n")

for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.metadata.get('source')}")
    print(f"    {doc.page_content[:200]}\n")

Query: What does the len() function do?
Retrieved 5 chunks

[1] https://docs.python.org/3/library/stdtypes.html
    Splitting and Joining
str.split()
str.rsplit()
bytes.split()
bytes.rsplit()

str.splitlines()
bytes.splitlines()

str.partition()
bytes.partition()

str.rpartition()
bytes.rpartition()

str.join()
byt

[2] https://docs.python.org/3/library/functions.html
    len(object, /)¶
Return the length (the number of items) of an object.  The argument may be a
sequence (such as a string, bytes, tuple, list, or range) or a collection
(such as a dictionary, set, or fr

[3] https://docs.python.org/3/library/stdtypes.html
    '0'
The conversion will be zero padded for numeric values.

'-'
The converted value is left adjusted (overrides the '0'
conversion if both are given).

' '
(a space) A blank should be left before a po

[4] https://docs.python.org/3/library/stdtypes.html
    len(s)
length of s


min(s)
smallest item of s


max(s)
largest item of s




Sequences of the same type also

## Step 7 — Build RAG Chain & Ask a Question

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-opus-4-6", temperature=0)

def ask(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

    messages = [
        {"role": "user", "content": (
            "You are an AI Documentation Assistant. "
            "Answer using ONLY the context below. "
            "Cite the source for every claim. "
            "If the answer is not in the context, say so.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}"
        )}
    ]

    response = llm.invoke(messages)
    return response.content

print("RAG function ready.")

## Step 8 — Ask Questions

In [ ]:
question = "What does the len() function do?"

answer = ask(question)

print(f"Q: {question}")
print(f"\nA: {answer}")

In [ ]:
question = "How does the map() function work?"

answer = ask(question)

print(f"Q: {question}")
print(f"\nA: {answer}")

In [ ]:
question = "What is the difference between a list and a tuple?"

answer = ask(question)

print(f"Q: {question}")
print(f"\nA: {answer}")